In [36]:
import pickle

with open('corpus/pubmed_ophthalmology/ophtho_abstracts.pkl', 'rb') as file:
    # Load the object from the file
    data = pickle.load(file)

In [37]:
len(data)

154591

In [43]:
import pickle
import os
from tqdm import tqdm
import json

def ends_with_ending_punctuation(s):
    ending_punctuation = ('.', '?', '!')
    return any(s.endswith(char) for char in ending_punctuation)

def concat(title, content):
    if ends_with_ending_punctuation(title.strip()):
        return title.strip() + " " + content.strip()
    else:
        return title.strip() + ". " + content.strip()

def chunk_dict(d, chunk_size):
    items = list(d.items())  # Convert dictionary to list of key-value pairs
    return [dict(items[i:i + chunk_size]) for i in range(0, len(items), chunk_size)]


data_list = chunk_dict(data, 1000)

if __name__ == "__main__":
    with open('corpus/pubmed_ophthalmology/ophtho_abstracts.pkl', 'rb') as file:
        # Load the object from the file
        data = pickle.load(file)
        
    if not os.path.exists("corpus/pubmed_ophthalmology/chunk"):
        os.makedirs("corpus/pubmed_ophthalmology/chunk")

    chunked_files = chunk_dict(data, 1000)
    fnames = ["pubmed_ophthalmology" + str(i) for i in range(len(chunked_files))]
    for i, chunk in tqdm(enumerate(chunked_files)):
        if os.path.exists(fnames[i]):
            continue
        # Organize chunk into required formula
        titles = [article['title'] for article in chunk.values()]
        abstracts = [article['abstract_text'] for article in chunk.values()]
        ids = list(chunk.keys())
        # titles, abstracts, ids = extract(gz_fpath)
        saved_text = [json.dumps({"id": fnames[i]+'_'+str(j), "title": titles[j], "content": abstracts[j], "contents": concat(titles[j], abstracts[j]), "PMID": str(ids[j])}) for j in range(len(titles))]
        # Save file
        with open("corpus/pubmed_ophthalmology/chunk/{:s}".format(fnames[i] + '.jsonl'), 'w') as f:
            f.write('\n'.join(saved_text))

155it [00:02, 57.62it/s]


In [ ]:
import json
import sys
import os
# Temporarily add MedRAG repo
sys.path.append(os.path.abspath("../MedRAG"))
from src.medrag import MedRAG
from tqdm import tqdm
import re
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder

def standardize_question(text):
    pattern = r"Question: (.*?)\n(A\..*?)(\nB\..*?)(\nC\..*?)(\nD\..*)"

    match = re.search(pattern, text, re.DOTALL)
    if match:
        question = match.group(1)
        choices = {
            "A": match.group(2).strip()[3:],  # Remove 'A. ' from the beginning
            "B": match.group(3).strip()[3:],  # Remove 'B. ' from the beginning
            "C": match.group(4).strip()[3:],  # Remove 'C. ' from the beginning
            "D": match.group(5).strip()[3:]   # Remove 'D. ' from the beginning
        }
        return question, choices
    else:
        print("[ERROR]: ", text)


def get_answer(cot, q, c):
    try:
        answer, _, _ = cot.answer(question=q, options={"A": c['A'], "B": c['B'], "C": c['C'], "D": c['D']})
    except: 
        return None
    try:
        answer = json.loads(answer)
        return answer['answer_choice']
    except:
        if isinstance(answer, str):
            list_of_string = re.findall(r'"(.*?)"', answer)
            if len(list_of_string) != 0:
                return list_of_string[-1]
            print('Output is not in json format or answer is not string!')
            print(answer)
            return None
        
            

model_names = ["OpenAI/gpt-35-turbo-16k", "OpenAI/gpt-4o"]
# model_names = ["OpenAI/gpt-4o"]
retriever_names = ["BM25", "MedCPT"]
# retriever_names = ["Contriever", "SPECTER"]
corpus_names = ["PubMed", "StatPearls", "Textbooks", "Wikipedia", "MedCorp"]
# corpus_names = ["MedCorp"]

with open('../data/results/jama_ophthalmology_clinical_challenge_378_result.json', 'r') as file:
    data = json.load(file)

# Test without RAG
for model_name_ in model_names:
    output_name_ = "JAMA(Ophthalmology)+" + model_name_ + "+CoT"
    print('---', output_name_, '---')
    cot = MedRAG(llm_name=model_name_, rag=False)
    for i, question in tqdm(enumerate(data)):
        if output_name_ in data[i].keys():
            continue
        else:
            q, c = standardize_question(question['query'])
            a = get_answer(cot, q, c)
            data[i][output_name_] = a
        with open('../data/results/jama_ophthalmology_clinical_challenge_378_result.json', 'w') as file:
                json.dump(data, file)

for model_name_ in model_names:
    for retriever_name_ in retriever_names:
        for corpus_name_ in corpus_names:
            output_name_ = "JAMA(Ophthalmology)+" + model_name_ + "+" + retriever_name_ + "+" + corpus_name_
            print('---', output_name_, '---')
            cot = MedRAG(llm_name=model_name_, rag=True, retriever_name=retriever_name_, corpus_name=corpus_name_)
            correct, incorrect = 0, 0
            unknown = 0
            true_labels = []
            predicted_labels = []
            for i, question in tqdm(enumerate(data)):
                if output_name_ in data[i].keys():
                    continue
                else:
                    q, c = standardize_question(question['query'])
                    a = get_answer(cot, q, c)
                    data[i][output_name_] = a

            with open('../data/results/jama_ophthalmology_clinical_challenge_378_result.json', 'w') as file:
                json.dump(data, file)

In [44]:
import sys
os.system(f'"{sys.executable}" -m pyserini.index.lucene --collection JsonCollection --input ./corpus/pubmed_ophthalmology/chunk --index ./corpus/pubmed_ophthalmology/index/bm25 --generator DefaultLuceneDocumentGenerator --threads 16')

2025-02-20 20:12:08,259 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:205) - Setting log level to INFO
2025-02-20 20:12:08,267 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:208) - ============ Loading Index Configuration ============
2025-02-20 20:12:08,267 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:209) - AbstractIndexer settings:
2025-02-20 20:12:08,267 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:210) -  + DocumentCollection path: ./corpus/pubmed_ophthalmology/chunk
2025-02-20 20:12:08,268 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:211) -  + CollectionClass: JsonCollection
2025-02-20 20:12:08,268 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:212) -  + Index path: ./corpus/pubmed_ophthalmology/index/bm25
2025-02-20 20:12:08,268 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:213) -  + Threads: 16
2025-02-20 20:12:08,269 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:214) -  + Optimize (merge s

Feb 20, 2025 8:12:08 PM org.apache.lucene.store.MMapDirectory lookupProvider


2025-02-20 20:12:08,501 INFO  [main] index.IndexCollection (IndexCollection.java:197) - IndexCollection settings:
2025-02-20 20:12:08,505 INFO  [main] index.IndexCollection (IndexCollection.java:198) -  + Generator: DefaultLuceneDocumentGenerator
2025-02-20 20:12:08,505 INFO  [main] index.IndexCollection (IndexCollection.java:199) -  + Language: en
2025-02-20 20:12:08,505 INFO  [main] index.IndexCollection (IndexCollection.java:200) -  + Stemmer: porter
2025-02-20 20:12:08,505 INFO  [main] index.IndexCollection (IndexCollection.java:201) -  + Keep stopwords? false
2025-02-20 20:12:08,505 INFO  [main] index.IndexCollection (IndexCollection.java:202) -  + Stopwords: null
2025-02-20 20:12:08,506 INFO  [main] index.IndexCollection (IndexCollection.java:203) -  + Store positions? false
2025-02-20 20:12:08,506 INFO  [main] index.IndexCollection (IndexCollection.java:204) -  + Store docvectors? false
2025-02-20 20:12:08,506 INFO  [main] index.IndexCollection (IndexCollection.java:205) -  + St

0

In [1]:
import json
import sys
import os
# Temporarily add MedRAG repo
# sys.path.append(os.path.abspath("../MedRAG"))
from src.medrag import MedRAG
from tqdm import tqdm
import re
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder

cot = MedRAG(llm_name="OpenAI/gpt-35-turbo-16k", rag=True, retriever_name="BM25", corpus_name="PubMed_Ophthalmology")
# cot = MedRAG(llm_name="OpenAI/gpt-35-turbo-16k", rag=True, retriever_name="BM25", corpus_name="PubMed")

/home/zc347/.conda/envs/bertpy311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/zc347/.conda/envs/bertpy311/lib/python3.11/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/home/zc347/.conda/envs/bertpy311/lib/python3.11/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/vast/palmer/home.mccleary/zc347/.conda/envs/bertpy311/lib/python3.11/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
Feb 20

In [2]:
from src.medrag import MedRAG

question = "A lesion causing compression of the facial nerve at the stylomastoid foramen will cause ipsilateral"
options = {
    "A": "paralysis of the facial muscles.",
    "B": "paralysis of the facial muscles and loss of taste.",
    "C": "paralysis of the facial muscles, loss of taste and lacrimation.",
    "D": "paralysis of the facial muscles, loss of taste, lacrimation and decreased salivation."
}

answer, _, _ = cot.answer(question=question, options=options)
print(f"Final answer in json with rationale: {answer}")

Final answer in json with rationale: {"step_by_step_thinking": "A lesion causing compression of the facial nerve at the stylomastoid foramen will result in paralysis of the facial muscles. Loss of taste, lacrimation, and decreased salivation are not typically associated with compression at this location.", "answer_choice": "A"}
